# A Particle Simulation Application

Run all cells from the project directory on a CUDA machine. The required GEM_2D case and the additional GEM_3D case each run on CPU and GPU using the same executable and input for both modes. GEM_3D has 56,623,104 particles and takes more time and memory.

Each execution creates a directory under **bonus_runs/** with hardware and toolchain details, the build log, four simulation logs and output directories, and a numerical summary for the report.

## Check the terminal environment

Run the Bash code cell below to check the working directory and CUDA tools. The first line, %%bash, tells Jupyter to execute the cell as a shell script. To run the same commands in a Bash terminal, copy the lines below %%bash.

In [ ]:
%%bash
set -e
pwd
ls -d Makefile src include inputfiles
nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
nvcc --version
g++ --version

In [ ]:
import json
import math
import os
import platform
import re
import shlex
import subprocess
from datetime import datetime
from pathlib import Path

project = Path.cwd().resolve()
arch = "sm_75"  # Change this to match the GPU compute capability if needed.
cases = ("GEM_2D", "GEM_3D")
atol, rtol = 1e-5, 1e-4

if not (project / "Makefile").is_file():
    raise FileNotFoundError("Start the notebook kernel in the iPIC3D-mini directory.")

run_root = project / "bonus_runs" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
run_root.mkdir(parents=True)
build = ["make", "-B", f"ARCH={arch}"]
executable = project / "bin" / "miniPIC.out"

cpuinfo = Path("/proc/cpuinfo")
cpu_model = re.search(r"^model name\s*:\s*(.+)$", cpuinfo.read_text(), re.MULTILINE) if cpuinfo.is_file() else None
environment = {
    "platform": platform.platform(),
    "cpu": cpu_model.group(1) if cpu_model else platform.processor(),
    "logical_cpus": os.cpu_count(),
    "cuda_architecture": arch,
    "build_command": shlex.join(build),
}
for name, command in {
    "gpu": ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total", "--format=csv"],
    "nvcc": ["nvcc", "--version"],
    "g++": ["g++", "--version"],
}.items():
    environment[name] = subprocess.check_output(command, text=True).strip()

(run_root / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
print(json.dumps(environment, indent=2))
print(f"Results directory: {run_root}")
os.environ["MINIPIC_RUN_ROOT"] = str(run_root)
os.environ["MINIPIC_CUDA_ARCH"] = arch


## Build and run from Bash

Run the Python setup cell above first to record the environment and create a fresh results directory. The next two Bash cells build once and run all four simulations. Each simulation runs in its own directory because the simulator writes VTK files under ./data/. Both modes use the same input and initialization, which seeds the random number generator with the species ID.

In [ ]:
%%bash
set -euo pipefail
: "${MINIPIC_RUN_ROOT:?Run the Python setup cell first}"
make -B "ARCH=$MINIPIC_CUDA_ARCH" 2>&1 | tee "$MINIPIC_RUN_ROOT/build.log"
test -x bin/miniPIC.out

In [ ]:
%%bash
set -euo pipefail
: "${MINIPIC_RUN_ROOT:?Run the Python setup cell first}"
project="$PWD"
for case in GEM_2D GEM_3D; do
    input="$project/inputfiles/$case.inp"
    test -f "$input"
    for mode in cpu gpu; do
        directory="$MINIPIC_RUN_ROOT/$case/$mode"
        mkdir -p "$directory/data"
        printf '\nRunning %s with %s mover\n' "$case" "$mode"
        (
            cd "$directory"
            MINIPIC_MOVER="$mode" "$project/bin/miniPIC.out" "$input" 2>&1 | tee simulation.log
        )
    done
done
find "$MINIPIC_RUN_ROOT" -name 'rho_net_*.vtk'

## Correctness and performance

For each case, compare the latest saved cycle after checking that both modes produced the same output cycles and grid metadata. A point passes when its absolute error is at most **atol + rtol × |CPU value|**. The tolerances are experiment settings. The output records both maximum absolute error and RMSE.

Speedup is CPU time divided by GPU time. GPU mover time includes per-call allocation, transfers, kernel execution, and synchronization. Each timing comes from one simulation run.

In [ ]:
def read_rhonet(path):
    with path.open() as vtk:
        header = {}
        for line in vtk:
            tokens = line.split()
            if tokens and tokens[0] in {"DIMENSIONS", "ORIGIN", "SPACING", "POINT_DATA", "SCALARS"}:
                header[tokens[0]] = tokens[1:]
            if tokens == ["LOOKUP_TABLE", "default"]:
                break
        else:
            raise ValueError(f"Missing scalar data in {path}")
        if header.get("SCALARS", [])[:1] != ["rhonet"]:
            raise ValueError(f"Expected rhonet scalars in {path}")
        shape = tuple(map(int, header["DIMENSIONS"]))
        origin = tuple(map(float, header["ORIGIN"]))
        spacing = tuple(map(float, header["SPACING"]))
        count = int(header["POINT_DATA"][0])
        values = [float(token) for line in vtk for token in line.split()]
    if len(shape) != 3 or count <= 0 or math.prod(shape) != count or len(values) != count:
        raise ValueError(f"Invalid grid or point count in {path}")
    if not all(map(math.isfinite, values)):
        raise ValueError(f"Non-finite charge density in {path}")
    return (shape, origin, spacing), values


def read_timings(log):
    labels = {
        "total_s": "Tot. Simulation Time",
        "mover_s_per_cycle": "Mover Time / Cycle",
        "interp_s_per_cycle": "Interp. Time / Cycle",
    }
    times = {}
    contents = log.read_text()
    for key, label in labels.items():
        match = re.search(rf"^\s*{re.escape(label)}\s*\(s\)\s*=\s*(\S+)\s*$",
                          contents, re.MULTILINE)
        if not match:
            raise ValueError(f"Missing {label} in {log}")
        value = float(match.group(1))
        if not math.isfinite(value) or value <= 0:
            raise ValueError(f"Invalid {label} in {log}: {value}")
        times[key] = value
    return times


def compare_case(case):
    snapshots = {}
    for mode in ("cpu", "gpu"):
        data = logs[case, mode].parent / "data"
        snapshots[mode] = {int(file.stem.split("_")[-1]): file for file in data.glob("rho_net_*.vtk")}
    if not snapshots["cpu"] or snapshots["cpu"].keys() != snapshots["gpu"].keys():
        raise ValueError(f"{case}: missing or different CPU/GPU output cycles")
    cycle = max(snapshots["cpu"])
    cpu_grid, cpu_values = read_rhonet(snapshots["cpu"][cycle])
    gpu_grid, gpu_values = read_rhonet(snapshots["gpu"][cycle])
    if cpu_grid != gpu_grid:
        raise ValueError(f"{case}: CPU and GPU grids differ")

    errors = [abs(gpu - cpu) for cpu, gpu in zip(cpu_values, gpu_values)]
    outside = sum(error > atol + rtol * abs(cpu) for error, cpu in zip(errors, cpu_values))
    cpu = read_timings(logs[case, "cpu"])
    gpu = read_timings(logs[case, "gpu"])
    return {
        "cycle": cycle,
        "grid_dimensions": cpu_grid[0],
        "points": len(errors),
        "rho_max_abs_error": max(errors),
        "rho_rmse": math.sqrt(math.fsum(error * error for error in errors) / len(errors)),
        "points_outside_tolerance": outside,
        "passed": outside == 0,
        "cpu": cpu,
        "gpu": gpu,
        "mover_speedup": cpu["mover_s_per_cycle"] / gpu["mover_s_per_cycle"],
        "total_speedup": cpu["total_s"] / gpu["total_s"],
    }

In [ ]:
logs = {
    (case, mode): run_root / case / mode / "simulation.log"
    for case in cases
    for mode in ("cpu", "gpu")
}
for (_, mode), log in logs.items():
    if f"Particle mover: {mode.upper()}" not in log.read_text().splitlines():
        raise RuntimeError(f"Could not verify mover selection in {log}")

results = {case: compare_case(case) for case in cases}
summary_path = run_root / "summary.json"
summary_path.write_text(json.dumps({"atol": atol, "rtol": rtol, "cases": results}, indent=2) + "\n")
for case, result in results.items():
    print(f"\n{case} | cycle {result['cycle']} | {result['points']:,} grid points")
    print(f"  rho_net: max abs error {result['rho_max_abs_error']:.6g}, "
          f"RMSE {result['rho_rmse']:.6g}, {result['points_outside_tolerance']:,} outside tolerance")
    print(f"  Correctness check: {'PASS' if result['passed'] else 'FAIL'}")
    for key, label in (("mover_s_per_cycle", "Mover (s/cycle)"),
                       ("interp_s_per_cycle", "Interpolation (s/cycle)"),
                       ("total_s", "Total (s)")):
        print(f"  {label}: CPU {result['cpu'][key]:.6g}, GPU {result['gpu'][key]:.6g}")
    print(f"  Speedup: mover {result['mover_speedup']:.3f}×, total {result['total_speedup']:.3f}×")
print(f"\nReport data: {summary_path}")